# SO-ARM101 · GitHub에서 받아 강화학습 실행하기

**Colab GPU · MuJoCo/MJWarp · PPO 직접 관절 제어 · 브라우저 3D 뷰어**

환경·학습·뷰어 코드는 GitHub 저장소의 `environment.py`, `learning.py`, `viewer.py`에 있습니다.
이 노트북은 저장소를 내려받고 설정과 실행을 담당합니다. 긴 `MODULE_SOURCE` 문자열은 포함하지 않습니다.

정책이 팔 관절과 집게의 목표 각도 변화량을 직접 선택합니다. 미리 정한 집기·운반 경로는 사용하지 않습니다.
바구니의 공 6개를 대응하는 2×3 칸에 옮기며, 먼저 놓은 공도 계속 물리 계산에 참여합니다.
초기 정책은 미학습 상태이며 학습 성능이나 실행 시간은 측정하지 않았습니다.

**제작 과정에서 실행·구문·물리·GPU·학습·뷰어 검증을 수행하지 않았습니다.**

Colab에서 GPU 런타임을 선택한 뒤 순서대로 실행하세요. 1번에는 `jjeong3150/so101-rl-pjt` 주소가 기본값으로 들어 있습니다.
특정 버전으로 실행하려면 `GITHUB_REF`에 브랜치·태그·커밋을 입력합니다. 비우면 저장소 기본 브랜치를 사용합니다.

## 1. GitHub 저장소 받기

`REPO_URL`에는 코드를 올린 저장소의 HTTPS 주소를 입력합니다. 아래 셀은 공개 저장소의 인증 없는 복제를 기준으로 합니다.
비공개 저장소는 Colab에서 접근할 수 있는 인증 설정이 별도로 필요합니다.
같은 런타임에서 다시 실행하면 내려받은 폴더를 사용합니다. `GITHUB_REF`를 지정하면 해당 버전을 가져옵니다.

In [ ]:
#@title 1. 저장소 주소를 입력하고 내려받기
REPO_URL = "https://github.com/jjeong3150/so101-rl-pjt.git" #@param {type:"string"}
GITHUB_REF = "main" #@param {type:"string"}

from pathlib import Path
import importlib
import json
import subprocess
import sys

if "runner" in globals() and runner.thread is not None and runner.thread.is_alive():
    raise RuntimeError("먼저 학습을 중지한 뒤 코드를 다시 불러오세요.")
if "viewer" in globals():
    viewer.close()
if not REPO_URL.strip():
    raise ValueError("REPO_URL에 사용할 GitHub 저장소 주소를 입력하세요.")
REPO_DIR = Path("/content/so101_mjwarp_rl")
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL.strip(), str(REPO_DIR)])
else:
    print("이미 내려받은 폴더를 사용합니다:", REPO_DIR)
if GITHUB_REF.strip():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF.strip()])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"])
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for module_name in ("environment", "learning", "viewer"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print("사용할 커밋:", REPO_COMMIT)

## 2. 패키지 설치

설치 목록은 저장소의 `requirements-colab.txt`에서 관리합니다.
다른 버전의 라이브러리를 이미 불러온 런타임이라면 새 GPU 런타임에서 시작하세요.

In [ ]:
#@title 2. 저장소의 설치 목록 사용
import importlib.util
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements-colab.txt")])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.9.1",
                           "--index-url", "https://download.pytorch.org/whl/cu126"])
print("패키지 설치 셀이 끝났습니다.")

## 3. Python 모듈 불러오기

아래 세 줄의 import가 저장소에 나눈 코드를 불러옵니다.
코드를 수정할 때는 저장소의 해당 Python 파일을 수정하면 됩니다.

In [ ]:
#@title 3. 환경·학습·뷰어 모듈 불러오기
importlib.invalidate_caches()
import copy
import json
import time
import numpy as np
import torch
import warp as wp
from google.colab import output
from IPython.display import display, Markdown, clear_output
from environment import SceneBundle, CPUEnv, WarpEnv, evaluate_policy
from learning import ActorCritic, PPORunner, DEFAULT_CONFIG
from viewer import LiveSO101Viewer
DEVICE = "cuda:0"
if not torch.cuda.is_available():
    raise RuntimeError("Colab의 GPU 런타임을 선택하세요.")
torch.cuda.set_device(0)
wp.init()
print("GPU:", torch.cuda.get_device_name(0))

## 4. 공식 로봇 모델 받기

원본은 MuJoCo Menagerie의 SO-ARM101 모델이며, 원본 커밋과 다운로드 파일 목록을 고정합니다.
팔의 구동력 제한은 원본 모델을 사용합니다. 공·바구니·격자 틀을 장면에 추가합니다.

In [ ]:
#@title 4-1. 저장소의 원본 모델 파일 목록
ASSET_MANIFEST = json.loads((REPO_DIR / "assets_manifest.json").read_text(encoding="utf-8"))

In [ ]:
#@title 4-2. 로봇 모델 다운로드
import concurrent.futures
import hashlib
import os
import urllib.request
from pathlib import PurePosixPath

ASSET_DIR = Path("/content/so101_assets")
ASSET_DIR.mkdir(parents=True, exist_ok=True)
commit = ASSET_MANIFEST["commit"]
base_url = f"https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/{commit}/"
def download_asset(entry):
    relative = PurePosixPath(entry["path"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("모델 폴더 안의 상대 경로가 필요합니다.")
    destination = ASSET_DIR.joinpath(*relative.parts)
    expected = entry["sha256"]
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() == expected:
        return
    request = urllib.request.Request(base_url + relative.as_posix(), headers={"User-Agent": "SO101-Colab-RL/1"})
    with urllib.request.urlopen(request, timeout=90) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise RuntimeError(f"다운로드 파일이 원본과 다릅니다: {relative}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".download")
    temporary.write_bytes(payload)
    os.replace(temporary, destination)
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(download_asset, ASSET_MANIFEST["files"]))
print("원본 모델 다운로드 완료:", commit)

## 5. 환경 수와 학습 설정

관측에는 관절 상태, 정책이 누적한 관절 목표, 손끝 위치, 공 위치·속도, 목표 위치와 채움 상태가 들어갑니다.
정책의 여섯 출력은 관절별 목표 각도 변화량으로 변환됩니다. 마지막 출력이 집게 개폐를 결정합니다.
보상은 공 접근, 물리적 집기·들기, 목표 접근, 안정적인 놓기와 전체 완료에 연결됩니다.
보상 계산이 관절 움직임이나 집게 개폐 명령을 대신 생성하지 않습니다.

기본 예산은 제한된 실습용 설정입니다. 성공률이 부족하면 저장한 정책으로 학습을 이어갈 수 있습니다.
`CHECKPOINT_PATH`는 선택 사항입니다. 이전 `latest.pt`를 Colab 파일 창에 올린 뒤 경로를 입력하세요.
이어 학습할 때 물리 에피소드와 난수 상태는 새로 시작하며, 중간 순간을 정확히 복원하는 기능은 아닙니다.

In [ ]:
#@title 5. 환경 수와 학습량 선택
ENV_PRESET = "자동" #@param ["자동", "4개", "8개", "16개", "32개"]
UPDATES = 1000 #@param {type:"integer"}
MAX_EPISODE_STEPS = 3000 #@param {type:"integer"}
EVAL_EVERY = 100 #@param {type:"integer"}
SEED = 7 #@param {type:"integer"}
CHECKPOINT_PATH = "" #@param {type:"string"}

if "runner" in globals() and runner.thread is not None and runner.thread.is_alive():
    raise RuntimeError("진행 중인 학습을 먼저 중지하세요.")
if "viewer" in globals():
    viewer.close()
gpu_name = torch.cuda.get_device_name(0).upper()
num_envs = 16 if "L4" in gpu_name else 8
if ENV_PRESET != "자동":
    num_envs = int(ENV_PRESET.removesuffix("개"))
config = dict(DEFAULT_CONFIG, num_envs=num_envs, horizon=128, updates=UPDATES,
              eval_every=EVAL_EVERY, eval_episodes=1, seed=SEED)
bundle = SceneBundle(ASSET_DIR, max_episode_steps=MAX_EPISODE_STEPS)
RUN_DIR = Path("/content/so101_direct_rl_runs") / time.strftime("%Y%m%d_%H%M%S", time.gmtime())
def make_training_env():
    return WarpEnv(bundle, num_envs=config["num_envs"], device=DEVICE, seed=config["seed"], jitter=.002)
def fixed_evaluator(policy, mode="learned", seed=2026):
    return evaluate_policy(bundle, policy, mode=mode, episodes=config["eval_episodes"], seed=seed, jitter=.002)
runner = PPORunner(config, make_env=make_training_env, evaluator=fixed_evaluator,
                   output_dir=RUN_DIR, device=DEVICE, observation_dim=bundle.observation_dim,
                   action_dim=bundle.action_dim,
                   metadata={"task": "six_balls_basket_to_2x3_grid_direct_policy",
                             "asset_manifest": ASSET_MANIFEST, "source_commit": REPO_COMMIT, "max_episode_steps": MAX_EPISODE_STEPS,
                             "action": "policy_selected_joint_target_increments"})
if CHECKPOINT_PATH.strip():
    runner.load_checkpoint(CHECKPOINT_PATH.strip(), restore_optimizer=True)
    print("이전 정책을 불러왔습니다. 업데이트 횟수는 이번에 추가할 학습량입니다.")
print("환경:", num_envs, "| 관측:", bundle.observation_dim, "| 행동:", bundle.action_dim)
print("이번 실행 전이 예산:", num_envs * config["horizon"] * config["updates"])
print("저장 위치:", RUN_DIR)

## 6. 웹 3D 뷰어

GPU 학습과 별개인 CPU 환경 한 대에서 정책을 실행합니다. 카메라 회전·확대, 일시정지, 전체 초기화를 사용할 수 있습니다.
학습 시작 시 정책 / 최신 정책 / 무작위 행동을 비교합니다. 정책은 전체 에피소드 시작 때 불러옵니다.
공 6개가 목표 칸에 안정적으로 남아 있을 때 전체 성공입니다. 기존에 넣은 공이 밀려나면 채움 수가 감소합니다.
초기에는 공을 집지 못하는 장면이 나올 수 있습니다.

In [ ]:
#@title 6-1. Colab 내부 뷰어
if "viewer" in globals():
    viewer.close()
viewer = LiveSO101Viewer(runner, bundle, port=8081, seed=2026, jitter=.002)
output.serve_kernel_port_as_iframe(viewer.server.get_port(), path="/", width="100%",
                                  height="700", cache_in_notebook=False)

In [ ]:
#@title 6-2. 본인 세션의 별도 브라우저 탭 링크
SHOW_OWN_TAB_LINK = True #@param {type:"boolean"}
if SHOW_OWN_TAB_LINK:
    try:
        import html
        from IPython.display import HTML
        private_viewer_url = output.eval_js(f"google.colab.kernel.proxyPort({viewer.server.get_port()})")
        if private_viewer_url:
            safe_url = html.escape(str(private_viewer_url), quote=True)
            display(HTML(f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer">내 3D 뷰어를 별도 탭에서 열기</a>'))
        else:
            print("별도 탭 주소를 가져오지 못했습니다. 내부 뷰어를 사용하세요.")
    except Exception:
        print("별도 탭 주소를 가져오지 못했습니다. 내부 뷰어를 사용하세요.")

별도 탭은 현재 로그인한 본인 Colab 세션의 프록시 주소를 사용합니다. 런타임을 열어 두세요.
브라우저 보안 설정에 따라 별도 탭 연결이 안 될 수 있으며, 그때는 내부 뷰어를 사용하세요.
외부에 공개하는 터널은 만들지 않습니다.

## 7. GPU 병렬 강화학습 시작

GPU 커널 준비 이후 경험 수집과 PPO 업데이트가 진행됩니다. 처음 평가가 끝나기까지 기다릴 수 있습니다.
학습과 웹 뷰어는 각각 별도의 스레드에서 진행됩니다.

In [ ]:
#@title 7. 학습 시작
runner.start()
print("학습을 시작했습니다. 아래 표시 셀을 중단해도 학습 스레드는 계속 실행됩니다.")

## 8. 학습 진행 보기

학습 보상, 평가 종료 때 채워진 칸 수, 전체 성공률과 PPO 손실을 구분해 읽으세요.
기본 평가는 고정된 한 조건의 결과이므로 다양한 초기 조건에 대한 성공률 추정은 아닙니다.
`DISPLAY_LIVE=False`로 바꾸면 현재 상태만 표시하고 바로 다음 셀로 넘어갑니다.
이 표시 셀의 중단은 학습 중단과 다릅니다. 학습 중단은 9번을 사용하세요.

In [ ]:
#@title 8. 보상과 채운 칸 수 보기
DISPLAY_LIVE = True #@param {type:"boolean"}
import matplotlib.pyplot as plt
def show_evaluation():
    baseline = runner.baseline or {}
    latest = runner.final_evaluation or {}
    rows = [("학습 시작 시 정책", baseline.get("learned")),
            ("최신 정책", latest.get("learned")),
            ("무작위 행동", latest.get("random"))]
    lines = ["| 평가 방식 | 전체 성공 | 채운 칸 | 평균 보상 |", "|---|---:|---:|---:|"]
    for label, result in rows:
        if result:
            lines.append(f"| {label} | {result.get('success_rate', 0):.0%} | {result.get('mean_filled_count', 0):.1f}/6 | {result.get('mean_return', 0):.1f} |")
        else:
            lines.append(f"| {label} | 대기 | — | — |")
    display(Markdown("\n".join(lines)))
try:
    while True:
        with runner.lock:
            history = copy.deepcopy(runner.history)
        clear_output(wait=True)
        print("상태:", runner.status, "| 업데이트:", runner.completed_updates, "| 누적 전이:", runner.total_steps)
        print("브라우저 접속:", len(viewer.server.get_clients()))
        show_evaluation()
        if history:
            x = [row["steps"] for row in history]
            fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
            axes[0].plot(x, [row.get("train_mean_return") for row in history])
            axes[0].set(xlabel="Transitions", ylabel="Episode return")
            axes[1].plot(x, [row.get("train_mean_filled_count") for row in history])
            axes[1].set(xlabel="Transitions", ylabel="Filled cells", ylim=(-.1, 6.1))
            axes[2].plot(x, [row.get("ppo_loss") for row in history])
            axes[2].set(xlabel="Transitions", ylabel="PPO loss")
            fig.tight_layout()
            display(fig)
            plt.close(fig)
        if runner.error:
            print(runner.error)
            break
        if viewer.error:
            print("뷰어 오류:", viewer.error)
        if not DISPLAY_LIVE or runner.thread is None or not runner.thread.is_alive():
            break
        time.sleep(3)
except KeyboardInterrupt:
    print("표시만 중단했습니다. 학습을 멈추려면 9번을 실행하세요.")

## 9. 중지와 정책 저장

체크하고 실행하면 중지를 요청합니다. 저장까지 끝난 뒤 결과를 다운로드하세요.
체크포인트는 정책과 최적화 정보를 보관합니다. 이어 학습할 때 물리 상태와 난수 흐름은 새로 시작합니다.

In [ ]:
#@title 9. 학습 중지 요청
STOP_TRAINING = False #@param {type:"boolean"}
if STOP_TRAINING:
    runner.stop_event.set()
    if runner.thread is not None:
        runner.thread.join(timeout=10)
    print("상태:", runner.status)
else:
    print("학습을 계속합니다. 중지하려면 체크하고 이 셀을 실행하세요.")
print("결과 폴더:", RUN_DIR)

## 10. 결과 다운로드

학습 완료나 중지 후 체크하여 실행합니다. 저장한 `latest.pt`를 다음 실행의 `CHECKPOINT_PATH`에 지정하면
정책과 최적화 상태를 불러와 추가 학습합니다. 물리 에피소드와 난수 흐름은 새로 시작합니다.

In [ ]:
#@title 10. 정책과 실행 코드 ZIP 다운로드
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    if runner.thread is None or runner.thread.is_alive():
        raise RuntimeError("학습 완료 또는 중지 후 다운로드하세요.")
    runner.save("latest.pt")
    included = RUN_DIR / "included_code"
    included.mkdir(parents=True, exist_ok=True)
    for name in ("environment.py", "learning.py", "viewer.py", "assets_manifest.json", "requirements-colab.txt"):
        shutil.copy2(REPO_DIR / name, included / name)
    (RUN_DIR / "source_commit.txt").write_text(REPO_COMMIT, encoding="utf-8")
    archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    files.download(archive)

In [ ]:
#@title 11. 뷰어 종료 — 선택
CLOSE_VIEWER = False #@param {type:"boolean"}
if CLOSE_VIEWER:
    viewer.close()
    print("뷰어를 닫았습니다. GPU 사용을 마치면 Colab 런타임도 연결 해제하세요.")

원본과 참고: [MuJoCo Menagerie SO-ARM101](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotstudio_so101),
[MuJoCo Warp](https://github.com/google-deepmind/mujoco_warp), [NVIDIA Warp](https://github.com/NVIDIA/warp),
[mjviser](https://github.com/mujocolab/mjviser), [Viser](https://github.com/viser-project/viser),
[PPO 논문](https://arxiv.org/abs/1707.06347).
원본 로봇 라이선스는 다운로드한 `robotstudio_so101/LICENSE`에 있습니다.

이 노트북은 시뮬레이션 학습용이며 실제 로봇을 연결하거나 움직이지 않습니다.